In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02_silver - Day 6: build the silver layer
# MAGIC Run each cell from top to bottom with **Shift + Enter**. Read the comments in each cell before running it.
# MAGIC
# MAGIC | Step | Builds |
# MAGIC | --- | --- |
# MAGIC | 1 | silver.orders + silver.orders_quarantine |
# MAGIC | 2 | silver.order_items + silver.order_items_quarantine |
# MAGIC | 3 | silver.customers |
# MAGIC | 4 | silver.reviews (one review per order) |
# MAGIC | 5 | Reconciliation for all 4 tables |

# COMMAND ----------

# CELL 0 - Setup
# Load Spark's built-in functions with the short name F.
# Every cell below uses F (F.col, F.when, F.trim ...), so this cell must run first.
from pyspark.sql import functions as F

# Load the Window tool, used in Step 4 to number rows inside groups (ROW_NUMBER).
from pyspark.sql.window import Window

# The date-and-time format used by every Olist date column, e.g. 2017-10-02 10:56:33.
# yyyy = year, MM = month, dd = day, HH = hour (00-23), mm = minutes, ss = seconds.
TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

print("Setup done")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 1 - silver.orders

# COMMAND ----------

# CELL 1 - Read bronze orders and convert the 5 date columns from text to timestamps

# Read the bronze table. Silver is always built from bronze, never from the raw CSV files.
orders_bronze = spark.table("workspace.bronze.orders")

# The 5 date columns, in a list so one loop can convert them all.
order_ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Standardise the status text: trim removes spaces at the ends, lower makes it lowercase,
# so ' Delivered' and 'delivered' become the same value.
orders_typed = orders_bronze.withColumn("order_status", F.lower(F.trim(F.col("order_status"))))

# Convert each date column. try_to_timestamp returns NULL (instead of crashing the job)
# if a value cannot be converted. F.lit wraps the format text as a column value.
for c in order_ts_cols:
    orders_typed = orders_typed.withColumn(c, F.try_to_timestamp(F.col(c), F.lit(TS_FORMAT)))

# Show column names and types. The 5 date columns should now say "timestamp".
orders_typed.printSchema()

# COMMAND ----------

# CELL 2 - Check that no dates silently failed to convert
# try_to_timestamp hides failures by returning NULL, so we count them ourselves:
# dates present in bronze minus dates present after converting = failed conversions.
for c in order_ts_cols:
    before = orders_bronze.filter(F.col(c).isNotNull()).count()
    after = orders_typed.filter(F.col(c).isNotNull()).count()
    print(f"{c:32} bronze {before:>7,}  typed {after:>7,}  failed {before - after}")

# Expected: failed 0 for every column.

# COMMAND ----------

# CELL 3 - Apply the 2 quality rules from Day 3, then split into silver and quarantine

# F.when(condition, label) works like IF; .when like ELSE IF; .otherwise like ELSE.
# In PySpark, & means AND, and each condition must be inside its own brackets.
orders_dq = (
    F.when((F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNull(),
           "delivered_no_date")
     .when((F.col("order_status") == "canceled") & F.col("order_delivered_customer_date").isNotNull(),
           "canceled_with_delivery")
     .otherwise(None)          # NULL label = the row is clean
)

# Add the label to every row.
orders_checked = orders_typed.withColumn("dq_issue", orders_dq)

# Clean rows: label is NULL. Drop the empty label column before saving.
orders_good = orders_checked.filter(F.col("dq_issue").isNull()).drop("dq_issue")

# Broken rows: label is not NULL. Keep the reason and record when they were caught.
orders_bad = (orders_checked
              .filter(F.col("dq_issue").isNotNull())
              .withColumn("_quarantined_at", F.current_timestamp()))

# Save both as Delta tables. overwrite = safe to rerun (idempotent).
orders_good.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.silver.orders")
orders_bad.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.silver.orders_quarantine")

print("silver.orders:", orders_good.count(), "  quarantine:", orders_bad.count())
display(orders_bad.groupBy("dq_issue").count())   # how many rows per reason

# COMMAND ----------

# CELL 4 - Reconcile orders: every bronze row must be in exactly one place
b = orders_bronze.count()
s = spark.table("workspace.silver.orders").count()
q = spark.table("workspace.silver.orders_quarantine").count()
print(f"bronze {b:,} = silver {s:,} + quarantine {q:,}  ->  {'OK' if b == s + q else 'MISMATCH'}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 2 - silver.order_items

# COMMAND ----------

# CELL 5 - Type the item columns (money as exact DECIMAL), apply the price rule, split

items_bronze = spark.table("workspace.bronze.order_items")

items_typed = (items_bronze
    # Item number 1, 2, 3 is a count, so make it a whole number. try_cast gives NULL if impossible.
    .withColumn("order_item_id", F.expr("try_cast(order_item_id AS INT)"))
    # Money must be EXACT: DECIMAL(10,2) = up to 10 digits, 2 after the point.
    # A DOUBLE would drift (0.1 + 0.2 = 0.30000000000000004).
    .withColumn("price", F.expr("try_cast(price AS DECIMAL(10,2))"))
    .withColumn("freight_value", F.expr("try_cast(freight_value AS DECIMAL(10,2))"))
    .withColumn("shipping_limit_date", F.try_to_timestamp(F.col("shipping_limit_date"), F.lit(TS_FORMAT))))

# Validity rules: a price must exist, convert correctly, and be greater than 0.
items_dq = (F.when(F.col("price").isNull(), "price_missing_or_invalid")
             .when(F.col("price") <= 0, "price_not_positive")
             .otherwise(None))

items_checked = items_typed.withColumn("dq_issue", items_dq)
items_good = items_checked.filter(F.col("dq_issue").isNull()).drop("dq_issue")
items_bad = (items_checked.filter(F.col("dq_issue").isNotNull())
             .withColumn("_quarantined_at", F.current_timestamp()))

items_good.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.silver.order_items")
items_bad.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.silver.order_items_quarantine")

print("silver.order_items:", items_good.count(), "  quarantine:", items_bad.count())
items_good.select("price", "freight_value").printSchema()   # should say decimal(10,2)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 3 - silver.customers

# COMMAND ----------

# CELL 6 - Clean customer text columns

customers_silver = (spark.table("workspace.bronze.customers")
    # City: remove end spaces and make lowercase, so ' Sao Paulo' = 'sao paulo'.
    .withColumn("customer_city", F.lower(F.trim(F.col("customer_city"))))
    # State codes are written in capitals (SP, RJ).
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
    # Postcode prefix: pad with zeros on the left to 5 characters (1151 -> 01151).
    # It stays TEXT because you never do maths on a postcode.
    .withColumn("customer_zip_code_prefix", F.lpad(F.trim(F.col("customer_zip_code_prefix")), 5, "0")))

customers_silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.silver.customers")

print("silver.customers:", customers_silver.count())
display(customers_silver.select("customer_city", "customer_state", "customer_zip_code_prefix").limit(5))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 4 - silver.reviews (window function)

# COMMAND ----------

# CELL 7 - Measure the problem first: do some orders have more than one review?

reviews_bronze = spark.table("workspace.bronze.reviews")

# Give the DataFrame a temporary name so we can query it with SQL.
reviews_bronze.createOrReplaceTempView("reviews_b")

# If distinct_orders is less than total_rows, some orders have several reviews.
# That would cause FAN-OUT (duplicated rows) when gold joins reviews to orders.
display(spark.sql("""
    SELECT COUNT(*)                  AS total_rows,
           COUNT(DISTINCT review_id) AS distinct_review_ids,
           COUNT(DISTINCT order_id)  AS distinct_orders
    FROM reviews_b
"""))

# COMMAND ----------

# CELL 8 - Keep only the latest review per order using ROW_NUMBER

reviews_typed = (reviews_bronze
    .withColumn("review_score", F.expr("try_cast(review_score AS INT)"))     # 1-5 as a number
    .withColumn("review_creation_date", F.try_to_timestamp(F.col("review_creation_date"), F.lit(TS_FORMAT)))
    .withColumn("review_answer_timestamp", F.try_to_timestamp(F.col("review_answer_timestamp"), F.lit(TS_FORMAT))))

# Window: group rows by order WITHOUT collapsing them, newest answer first inside each order.
w = Window.partitionBy("order_id").orderBy(F.col("review_answer_timestamp").desc())

reviews_silver = (reviews_typed
    .withColumn("rn", F.row_number().over(w))   # number rows 1, 2, 3 ... inside each order
    .filter(F.col("rn") == 1)                    # keep only the newest review per order
    .drop("rn"))                                 # remove the helper column

reviews_silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.silver.reviews")

rb, rs = reviews_bronze.count(), reviews_silver.count()
print(f"bronze {rb:,}  silver {rs:,}  duplicates removed {rb - rs:,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 5 - Reconcile all silver tables

# COMMAND ----------

# CELL 9 - Prove that no rows were lost or created

def n(table_name):
    """Count the rows in any table."""
    return spark.table(table_name).count()

checks = [
    ("orders", n("workspace.bronze.orders"),
     n("workspace.silver.orders") + n("workspace.silver.orders_quarantine")),
    ("order_items", n("workspace.bronze.order_items"),
     n("workspace.silver.order_items") + n("workspace.silver.order_items_quarantine")),
    ("customers", n("workspace.bronze.customers"), n("workspace.silver.customers")),
]

for name, bronze_n, silver_n in checks:
    status = "OK" if bronze_n == silver_n else "MISMATCH"
    print(f"{name:12} bronze {bronze_n:>7,}  silver+quarantine {silver_n:>7,}  {status}")

# Reviews: the difference is EXPECTED - those are the duplicate reviews removed on purpose.
print(f"{'reviews':12} bronze {n('workspace.bronze.reviews'):>7,}  silver {n('workspace.silver.reviews'):>7,}  "
      f"(difference = duplicates removed on purpose)")

Setup done
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

order_purchase_timestamp         bronze  99,441  typed  99,441  failed 0
order_approved_at                bronze  99,281  typed  99,281  failed 0
order_delivered_carrier_date     bronze  97,658  typed  97,658  failed 0
order_delivered_customer_date    bronze  96,476  typed  96,476  failed 0
order_estimated_delivery_date    bronze  99,441  typed  99,441  failed 0
silver.orders: 99427   quarantine: 14


dq_issue,count
canceled_with_delivery,6
delivered_no_date,8


bronze 99,441 = silver 99,427 + quarantine 14  ->  OK
silver.order_items: 112650   quarantine: 0
root
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)

silver.customers: 99441


customer_city,customer_state,customer_zip_code_prefix
franca,SP,14409
sao bernardo do campo,SP,09790
sao paulo,SP,01151
mogi das cruzes,SP,08775
campinas,SP,13056


total_rows,distinct_review_ids,distinct_orders
99224,98410,98673


bronze 99,224  silver 98,673  duplicates removed 551
orders       bronze  99,441  silver+quarantine  99,441  OK
order_items  bronze 112,650  silver+quarantine 112,650  OK
customers    bronze  99,441  silver+quarantine  99,441  OK
reviews      bronze  99,224  silver  98,673  (difference = duplicates removed on purpose)
